# Generacion del dataset sintetico de ∂PV para el benchmark PINN

Dataset de referencia de la propuesta r04 (Seccion 7): cuatro familias de celdas (E1 homounion, E2 CdS/CdTe, E3 perovskita, E4 CdS/CdTe + LDS) y dos etapas por familia (A: dopajes [+ $\eta_{PL}$]; B: + espesores).

**Por bloques, para sumar sesiones sin repetir.** Cada familia-etapa tiene una sola secuencia Sobol fija de $2^{17}$ dispositivos, partida en bloques de $1024$. La unidad `"E1A-b003"` es el bloque 3 de E1A y produce `dpv_E1A_b003.h5`. Bloques distintos nunca comparten dispositivos, asi que cada sesion nueva suma datos. El bloque 0 lleva ademas las $2^{d_\xi}$ esquinas, y cada bloque re-simula un $5\%$ con $4\times$ nodos (piso de discretizacion, Ec. 21).

**Como se ejecuta**
- **Bajo `/remote-execution`**: `FORGE_RUN_MODE=smoke` corre el test rapido de la plataforma marcada. Si no, `FORGE_RUN_UNITS` trae los bloques de la sesion, y al terminar queda `resumen_sesion.json` con los bloques completos y los que hay que repetir.
- **A mano**: ajusta `MODO`, `UNIDADES` y `PLATAFORMA` en la celda de configuracion.


In [ ]:
#!/usr/bin/env python3
"""The first code cell of every notebook a job may run — copied byte for byte.

One question, answered once: WHERE IS THE REPOSITORY THIS NOTEBOOK RUNS
AGAINST. Every later cell reads `ROOT` and none of them asks again.

Opened by a person on their own machine, this answers it exactly the way
these notebooks always have. A notebook lives at `<repo>/<Name>/Notebooks/`,
so the repository is two directories above the working directory. Nothing was
handed over, nothing is checked, and the behaviour is unchanged.

Started by a runner on a remote worker, it does not answer it by looking
around. The runner exports the directory it cloned the pinned commit into and
the commit it pinned; this cell reads both, and PROVES the checkout is at that
commit before returning it.

Guessing is the whole reason this cell exists. Under the runner the kernel's
working directory is the runner's own and the clone sits one level inside it,
so two directories up is two levels ABOVE the working directory — a directory
that EXISTS on any worker. The path resolves, the insert succeeds, and the run
dies later with a missing module naming a package, never with the wrong root.
The one fact worth having is the one the failure never mentions.

Three refusals, and each one is a refusal rather than a fallback because this
is the path that spends metered quota:

- **Half a handoff** — one variable present and the other missing. Something
  built this environment and got it half right; that is precisely the state a
  fallback cannot tell apart from a laptop, and the fallback resolves a
  directory that exists everywhere.
- **A root that is not a checkout** — the declared directory is missing, or
  holds no readable `HEAD`. There is nothing to compare against, and an
  unproven root is what this cell was written to stop being acceptable.
- **A checkout at a different commit** — the declared commit and the one on
  disk disagree. A job that runs the wrong commit RUNS: it returns numbers
  shaped exactly like the right ones, and nothing downstream can tell.

Absent means LOCAL. It never means "work it out".

Importable and independently testable, the way the runner's own two cells
are: every function is pure and takes its environment and its working
directory as arguments, and only the last line — the binding a notebook cell
exists to perform — reads the real ones.
"""
import os
from pathlib import Path

#: The directory a runner cloned the pinned commit into. Forge-owned and
#: deliberately generic: this is the contract between a runner and the
#: notebook it starts, not a name borrowed from any one repository.
CLONE_ROOT_ENV = "FORGE_CLONE_ROOT"

#: The commit that clone was pinned to, exported beside it so this cell can
#: check rather than trust. A root on its own would only move the guess one
#: step: a directory handed over is still a directory nobody proved.
CLONE_COMMIT_ENV = "FORGE_CLONE_COMMIT"

#: How far above a notebook's own directory the repository sits when nobody
#: hands anything over: `<repo>/<Name>/Notebooks` -> `<repo>`.
LOCAL_ROOT_DEPTH = 1


def head_commit(root):
    """The commit `root`'s `HEAD` names, read straight out of `.git`.

    No subprocess, deliberately. A notebook cell that shells out needs a git
    binary on a worker's PATH that nothing here declared, and every checker
    that reads these notebooks then has to decide whether running the cell is
    safe — a cell that binds the repository is the last one that may be
    skipped for that reason.

    A pinned checkout is detached: the runner fetches a commit and checks out
    what it fetched, never a branch, so `HEAD` holds the raw commit and this
    is a one-line read. A symbolic `HEAD` is resolved anyway — through the
    loose ref and then `packed-refs` — so pointing these variables at an
    ordinary checkout gets an answer instead of a refusal that would be about
    the file format rather than about the commit.

    Returns `None` when there is nothing to read. The caller turns that into
    a refusal; this function never decides.
    """
    git = Path(root) / ".git"
    if git.is_file():
        pointer = git.read_text(encoding="utf-8").strip()
        if not pointer.startswith("gitdir:"):
            return None
        git = Path(root) / pointer[len("gitdir:"):].strip()
    if not git.is_dir():
        return None
    head = git / "HEAD"
    if not head.is_file():
        return None
    text = head.read_text(encoding="utf-8").strip()
    if not text.startswith("ref:"):
        return text or None
    ref = text[len("ref:"):].strip()
    loose = git / ref
    if loose.is_file():
        return loose.read_text(encoding="utf-8").strip() or None
    packed = git / "packed-refs"
    if packed.is_file():
        for line in packed.read_text(encoding="utf-8").splitlines():
            if line.startswith("#") or line.startswith("^"):
                continue
            fields = line.split()
            if len(fields) == 2 and fields[1] == ref:
                return fields[0]
    return None


def resolve_repository_root(environ=None, cwd=None, head_reader=head_commit):
    """The repository this notebook runs against, or a refusal saying why.

    `environ` and `cwd` are arguments so the whole decision can be driven
    without a kernel, a clone or a worker; the binding at the bottom of this
    cell passes the real ones.
    """
    environ = os.environ if environ is None else environ
    here = Path.cwd() if cwd is None else Path(cwd)
    declared_root = environ.get(CLONE_ROOT_ENV)
    declared_commit = environ.get(CLONE_COMMIT_ENV)

    if not declared_root and not declared_commit:
        return here.parents[LOCAL_ROOT_DEPTH]

    if not declared_root or not declared_commit:
        raise RuntimeError(
            "half a handoff: {0}={1!r} and {2}={3!r}. Both name the pinned "
            "checkout this notebook must run against, and one without the "
            "other is an environment somebody built and got half right. "
            "Falling back to the local layout here would resolve a directory "
            "that exists on any machine and is not this repository.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))

    root = Path(declared_root)
    found = head_reader(root) if root.is_dir() else None
    if found is None:
        raise RuntimeError(
            "{0}={1!r} is not a readable checkout: no commit could be read "
            "from its HEAD. {2} declares {3!r}, and there is nothing here to "
            "check it against.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))
    if found.lower() != declared_commit.lower():
        raise RuntimeError(
            "the checkout at {0}={1!r} is at commit {2}, and {3} declares "
            "{4}. A job that runs a commit nobody asked for RUNS, and the "
            "numbers it returns look exactly like the right ones.".format(
                CLONE_ROOT_ENV, declared_root, found,
                CLONE_COMMIT_ENV, declared_commit))
    return root


ROOT = resolve_repository_root()

In [ ]:
import os, sys, json
sys.path.insert(0, str(ROOT))

MODO_FORGE = os.environ.get("FORGE_RUN_MODE")              # "smoke" en el ensayo
UNIDADES_FORGE = os.environ.get("FORGE_RUN_UNITS")         # JSON con la lista de bloques
BAJO_RUNNER = os.environ.get("FORGE_CLONE_ROOT") is not None

# ---- Configuracion (bajo el runner, MODO y UNIDADES vienen del runner) ----
MODO = "TEST"                      # "TEST" o "PRODUCCION"
UNIDADES = ["E1A-b000"]            # bloques de 1024 dispositivos, p.ej. ["E1A-b000", "E1A-b001"]
PLATAFORMA = "cpu"                 # bandera: "cpu" (decidido con el test) o "gpu"
N_TRABAJADORES = None              # None = un proceso por nucleo (cpu) o por GPU
PRESUPUESTO_HORAS = 11.0           # Kaggle corta la sesion a las 12 h

if BAJO_RUNNER:
    MODO = "TEST" if MODO_FORGE == "smoke" else "PRODUCCION"
    if UNIDADES_FORGE is not None:
        UNIDADES = json.loads(UNIDADES_FORGE)
    DIR_SALIDA = os.getcwd()
else:
    DIR_SALIDA = str(ROOT / "salidas")
    os.makedirs(DIR_SALIDA, exist_ok=True)
print(f"MODO={MODO} UNIDADES={UNIDADES} PLATAFORMA={PLATAFORMA} salida en {DIR_SALIDA}")


In [ ]:
# deltapv fijado al commit verificado; --no-deps para no reemplazar el jax con CUDA de la imagen.
!pip install -q --no-deps "git+https://github.com/romanodev/deltapv@f874e24"
import h5py, scipy, numpy
print("h5py", h5py.__version__, "| scipy", scipy.__version__, "| numpy", numpy.__version__)

In [ ]:
from dpvgen import orquestador

orquestador.configurar(PLATAFORMA=PLATAFORMA, N_TRABAJADORES=N_TRABAJADORES,
                       PRESUPUESTO_HORAS=PRESUPUESTO_HORAS, DIR_SALIDA=DIR_SALIDA)
if MODO == "TEST":
    RESUMEN = orquestador.ejecutar_test_rapido(PLATAFORMA, 20)
elif MODO == "PRODUCCION":
    RESUMEN = orquestador.ejecutar_unidades(UNIDADES)
else:
    raise ValueError(MODO)


In [ ]:
# Verificacion de lo producido: tasa de cada estado (r04 exige reportarla) y PCE.
import glob
import numpy as np
from collections import Counter

for ruta in sorted(glob.glob(os.path.join(DIR_SALIDA, "*.h5"))):
    with h5py.File(ruta, "r") as f:
        print(f"\n{os.path.basename(ruta)}  familia={f.attrs['familia']} etapa={f.attrs['etapa']}  "
              f"{os.path.getsize(ruta)/1e9:.2f} GB")
        print("  xi:", [p["nombre"] for p in json.loads(f.attrs["parametros_xi"])])
        for g in ("sims", "esquinas", "refinados"):
            est = Counter(f[g][k].attrs["estado"] for k in f[g])
            n = sum(est.values())
            print(f"  /{g}: {n}  " + ", ".join(f"{e}={c} ({100*c/max(n,1):.1f}%)" for e, c in est.items()))
        oks = [k for k in f["sims"] if f["sims"][k].attrs["estado"] == "ok"]
        if oks:
            eff = np.array([f["sims"][k].attrs["eff"] for k in oks]) * 100
            nv = np.array([f["sims"][k].attrs["n_voltajes"] for k in oks])
            print(f"  PCE %: p5={np.percentile(eff,5):.2f} p50={np.median(eff):.2f} "
                  f"p95={np.percentile(eff,95):.2f} | voltajes por curva: {nv.min()}-{nv.max()}")